# Statistiques Descriptives

- **Objectif :** 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import CLEAN_INSCRIPTIONS 
from tableone import TableOne
from scipy.stats import mannwhitneyu, chi2_contingency, false_discovery_control

## Preparation des donnees

1. Chargement des donnees

In [ ]:
data = pd.read_csv(CLEAN_INSCRIPTIONS)
data.head()

2. Exclusion des variables redendantes

In [ ]:
redundant_cols = ['id_programme', 'resultat_final']
data.drop(redundant_cols, axis='columns', inplace=True)
data.head()

3. Type des variables

In [ ]:
num = ['nb_inscriptions_precedentes','age']
cat = ['genre','region','situation_handicap','niveau_scolaire','annee_scolaire','cohorte']
date = ['date_debut','date_annulation']
ids = ['id_inscription','id_etudiant']
churn = 'churn'

## Description des donnees 

1. Les variables numeriques

In [ ]:
data[num].describe()

In [ ]:
# Boxplot
for col in num:
    sns.boxplot(data=data, x=churn, y=col)
    plt.show()

2. Les variables Categoriques

In [ ]:
print(data[churn].value_counts(normalize=True))
for col in cat:
    print(f'Pour la variable {col}:')
    print(data[col].value_counts())
    print(pd.crosstab(data[col],data['churn'],normalize='index'))
    print('\n')


## Significativite des variables

In [ ]:
cols = num + cat 
table = TableOne(
    data,
    columns=cols,
    nonnormal=num,
    categorical=cat,
    groupby=churn,
    pval=True,
    htest_name=True
)
table

1. Test de Mann-Whitney pour les variables numeriques

> **$H_0$:** la distribution de la variable est identique entre les etudiants qui abandonnent (churn=1) et ceux qui n'abandonnent pas (churn=0).

> **$H_1$:** les deux distributions different (l'un des deux groupes tend a avoir des valeurs plus elevees que l'autre).

In [ ]:
res = []
for col in num:
    group0=data.loc[data[churn]==0,col]
    group1=data.loc[data[churn]==1,col]
    n0,n1=len(group0), len(group1)

    U,p = mannwhitneyu(group0,group1,alternative='two-sided')
    score = U/(n0*n1) #P(valeur churn=0 > valeur churn=1)
    res.append({
        'variable': col,
        'test': 'Mann-Whitney',
        'statistique': U,
        'p_value': p,
        'score' : score,
    })

    num_results = pd.DataFrame(res)


In [ ]:
num_results

2. Test de Chi-2 pour les variables categoriques

> **$H_0$:** la variable et churn sont independants (la proportion de churn est la meme dans toutes les categories de la variable).

> **$H_1$:** la variable et churn sont associes (la proportion de churn differe selon la categorie).

In [ ]:
def cramersV(contingency_table):
    chi2,p,dof,expected = chi2_contingency(contingency_table)
    n = contingency_table.values.sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt(chi2/(n*min_dim))
    return chi2,p,v
cat_res = []
for col in cat:
    table = pd.crosstab(data[col],data[churn])
    chi2,p,v = cramersV(table)
    cat_res.append({
        'variable': col,
        'test': 'Chi-2',
        'statistique': chi2,
        'p_value': p,
        'score' : v,
    })

    cat_results = pd.DataFrame(cat_res)


In [ ]:
cat_results

3. Correction de Benjamini-Hochberg

In [ ]:
# Ajustement des p_values
results = pd.concat([num_results, cat_results],ignore_index=True)
results['p_value_adj'] = false_discovery_control(results['p_value'],method='bh')
results.sort_values('p_value_adj')

4. Interpretation

Apres correction, `genre` (p_adj=0.102) et `cohorte` (p_adj=0.531) restent non significatifs.
Toutes les autres variables restent significatives apres correction.
`age` et `nb_inscriptions_precedentes` ont un p_value_adj tres faible, mais leur score est proche de 0.5 (0.533 et 0.485), ce qui signifie une capacite de discrimination quasi nulle entre churn=0 et churn=1. A n=12 888, un ecart infime suffit a produire un p-value tres faible sans que cela reflete un effet reel.
Pour les variables categoriques, comme churn n'a que 2 modalites, le V de Cramer suit directement les seuils de Cohen (negligeable <0.1, faible 0.1-0.3, moyen 0.3-0.5). Meme la variable la plus discriminante, `niveau_scolaire` (V=0.176), reste dans la zone "faible". `region`, `situation_handicap` et `annee_scolaire` sont sous le seuil de 0.1.
